# Module 7: Full RAG Pipeline Lab

This lab guides you through building a production-grade RAG system using **Microsoft AI Agents SDK** with an educational UI that exposes retrieval internals.

## Learning Objectives

By the end of this lab, you will be able to:
- Configure and use the Microsoft Azure AI Agents SDK
- Implement query decomposition and multi-hop reasoning
- Build agentic RAG systems with configurable retrieval
- Visualize the retrieval process for educational purposes

## Prerequisites

- Completed Module 0 (Azure resource setup)
- Backend and frontend services running (see `docker-compose.yaml`)
- `.env` file configured with Azure credentials

## 1. Setup & Environment Check

First, let's install the required packages and verify our environment.

In [ ]:
# Install required packages (run once)
%pip install httpx pandas matplotlib python-dotenv rich --quiet

In [ ]:
import httpx
import pandas as pd
import json
import os
from pathlib import Path
from dotenv import load_dotenv
from rich import print as rprint
from rich.table import Table
from rich.console import Console

# Load environment variables
load_dotenv("../../.env")

# Configuration
BASE_URL = "http://localhost:8000"
API_KEY = os.getenv("API_KEY", "")

# Create HTTP client with headers
client = httpx.Client(
    base_url=BASE_URL,
    headers={"X-API-Key": API_KEY},
    timeout=60.0
)

console = Console()
print("✅ Environment loaded successfully!")

In [ ]:
# Verify backend is running
try:
    response = client.get("/health")
    if response.status_code == 200:
        health = response.json()
        print(f"✅ Backend is healthy: {health}")
    else:
        print(f"⚠️ Backend returned status {response.status_code}")
except httpx.ConnectError:
    print("❌ Cannot connect to backend. Make sure it's running with:")
    print("   cd modules/module-7-pipeline && bash run_all.sh")
    print("   (or separately: bash run_backend.sh && bash run_frontend.sh)")

## 2. Exercise 1: Upload Documents

Let's upload some sample documents and track their processing status.

In [ ]:
# Find sample documents
sample_pdfs = list(Path("../../data/sample-pdfs").glob("*.pdf"))
sample_office = list(Path("../../data/sample-office").glob("*.*"))

print(f"Found {len(sample_pdfs)} PDFs and {len(sample_office)} Office files")
for f in sample_pdfs[:3]:
    print(f"  - {f.name}")

In [ ]:
# Upload a document
def upload_document(file_path: Path) -> dict:
    """Upload a document to the RAG pipeline."""
    with open(file_path, "rb") as f:
        files = {"file": (file_path.name, f, "application/pdf")}
        response = client.post("/api/documents/upload", files=files)
    return response.json()

# Upload the first PDF (if available)
if sample_pdfs:
    result = upload_document(sample_pdfs[0])
    doc_id = result.get("id")
    print(f"📤 Uploaded: {sample_pdfs[0].name}")
    print(f"   Document ID: {doc_id}")
    print(f"   Status: {result.get('status')}")

In [ ]:
# Check processing status (poll until complete)
import time

def wait_for_processing(doc_id: str, max_wait: int = 60):
    """Wait for document processing to complete."""
    for i in range(max_wait // 2):
        response = client.get(f"/api/documents/{doc_id}/status")
        status = response.json()
        
        if status["status"] == "completed":
            print(f"✅ Processing complete!")
            print(f"   Chunks created: {status.get('chunks_created', 'N/A')}")
            print(f"   Figures extracted: {status.get('figures_extracted', 'N/A')}")
            return status
        elif status["status"] == "failed":
            print(f"❌ Processing failed: {status.get('error_message')}")
            return status
        else:
            print(f"⏳ Status: {status['status']}...")
            time.sleep(2)
    
    print("⚠️ Timeout waiting for processing")
    return None

# Wait for the uploaded document
if doc_id:
    status = wait_for_processing(doc_id)

## 3. Exercise 2: Inspect Index Schema

Understanding the index schema is crucial for effective retrieval tuning.

In [ ]:
# Get index schema
response = client.get("/api/index/schema")
schema = response.json()

# Display fields as a table
table = Table(title="Index Fields")
table.add_column("Field Name", style="cyan")
table.add_column("Type", style="green")
table.add_column("Key", style="yellow")
table.add_column("Dimensions", style="magenta")

for field in schema.get("fields", []):
    table.add_row(
        field["name"],
        field["type"],
        "✓" if field.get("key") else "",
        str(field.get("dimensions", "")) if field.get("dimensions") else ""
    )

console.print(table)

In [ ]:
# Get index statistics
response = client.get("/api/index/stats")
stats = response.json()

print(f"📊 Index Statistics")
print(f"   Total documents: {stats.get('document_count', 0)}")
print(f"\n   Content Type Distribution:")
for ctype, count in stats.get("content_type_counts", {}).items():
    print(f"     - {ctype}: {count}")

## 4. Exercise 3: Basic Hybrid Query

Execute a simple RAG query and inspect the retrieved chunks.

In [ ]:
def execute_query(question: str, config: dict = None) -> dict:
    """Execute a RAG query with optional config override."""
    payload = {"question": question}
    if config:
        payload["config"] = config
    
    response = client.post("/api/query", json=payload)
    return response.json()

def display_query_result(result: dict):
    """Display query result in a readable format."""
    print(f"📝 Answer:\n{result['answer']}\n")
    print(f"⏱️ Retrieval Time: {result['metadata']['retrieval_time_ms']}ms")
    print(f"🎯 Strategy Used: {result['metadata']['strategy_used']}")
    print(f"📦 Chunks Retrieved: {result['metadata']['total_chunks_retrieved']}")
    
    print("\n📚 Sources:")
    for i, source in enumerate(result['sources'][:5], 1):
        print(f"  {i}. [{source['content_type']}] {source['source_document']} (p.{source['page_numbers']})")
        print(f"     Score: {source['relevance_score']:.3f}")
        print(f"     Preview: {source['content'][:100]}...")

In [ ]:
# Execute a basic hybrid search query
question = "What are the main specifications of Metro Station 36?"

result = execute_query(question, config={
    "search_mode": "hybrid",
    "top_k": 5,
    "retrieval_strategy": "hybrid"
})

display_query_result(result)

## 5. Exercise 4: Compare Retrieval Strategies

Compare how different retrieval strategies handle the same query.

In [ ]:
# Compare hybrid vs agentic retrieval
complex_question = "What is the ventilation system design for Metro Station 36, and what safety measures are in place for emergency scenarios?"

strategies = ["hybrid", "agentic"]
results = {}

for strategy in strategies:
    print(f"\n{'='*50}")
    print(f"🔄 Testing strategy: {strategy.upper()}")
    print('='*50)
    
    result = execute_query(complex_question, config={
        "retrieval_strategy": strategy,
        "top_k": 5
    })
    results[strategy] = result
    
    print(f"⏱️ Time: {result['metadata']['retrieval_time_ms']}ms")
    print(f"📦 Chunks: {result['metadata']['total_chunks_retrieved']}")
    
    # Show query decomposition for agentic
    if result['metadata'].get('query_decomposition'):
        print(f"🔀 Sub-queries:")
        for sq in result['metadata']['query_decomposition']['sub_queries']:
            print(f"   - {sq['query']} ({sq['results_count']} results)")

In [ ]:
# Visualize retrieval comparison
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Time comparison
ax1 = axes[0]
times = [results[s]['metadata']['retrieval_time_ms'] for s in strategies]
ax1.bar(strategies, times, color=['steelblue', 'coral'])
ax1.set_ylabel('Time (ms)')
ax1.set_title('Retrieval Time Comparison')

# Chunks comparison
ax2 = axes[1]
chunks = [results[s]['metadata']['total_chunks_retrieved'] for s in strategies]
ax2.bar(strategies, chunks, color=['steelblue', 'coral'])
ax2.set_ylabel('Chunks Retrieved')
ax2.set_title('Chunks Retrieved Comparison')

plt.tight_layout()
plt.show()

## 6. Exercise 5: Tune Retrieval Parameters

Experiment with different parameter combinations to see their effect on results.

In [ ]:
# Test different top_k values
question = "What are the operating pressure ranges and safety limits?"

top_k_values = [1, 3, 5, 10]
top_k_results = []

for k in top_k_values:
    result = execute_query(question, config={"top_k": k})
    top_k_results.append({
        "top_k": k,
        "time_ms": result['metadata']['retrieval_time_ms'],
        "answer_length": len(result['answer']),
        "sources_count": len(result['sources'])
    })

df = pd.DataFrame(top_k_results)
print(df.to_string(index=False))

In [ ]:
# Test content type filtering
question = "Show me the technical diagrams and data tables in the documentation"

filters = ["all", "table", "figure"]
filter_results = {}

for f in filters:
    result = execute_query(question, config={
        "content_type_filter": f,
        "top_k": 5
    })
    filter_results[f] = result
    
    print(f"\n🔍 Filter: {f}")
    print(f"   Chunks: {len(result['sources'])}")
    if result['sources']:
        types = [s['content_type'] for s in result['sources']]
        print(f"   Types: {set(types)}")

## 7. Exercise 6: Analyze Query Decomposition (Agentic Mode)

When using agentic retrieval, the AI Agent decomposes complex queries into sub-queries. Let's analyze this process.

In [ ]:
# Execute a complex multi-part question with agentic mode
complex_question = """
Compare the compressor specifications across the documentation.
What are the recommended maintenance intervals, and what happens
if the discharge temperature exceeds the safety threshold?
"""

result = execute_query(complex_question, config={
    "retrieval_strategy": "agentic",
    "top_k": 5
})

# Display decomposition details
metadata = result['metadata']
print(f"🧠 Original Query: {metadata.get('query_decomposition', {}).get('original_query', 'N/A')}\n")

if metadata.get('query_decomposition'):
    decomp = metadata['query_decomposition']
    print(f"🔀 Decomposed into {len(decomp['sub_queries'])} sub-queries:\n")
    
    for i, sq in enumerate(decomp['sub_queries'], 1):
        print(f"  {i}. {sq['query']}")
        print(f"     Intent: {sq.get('intent', 'N/A')}")
        print(f"     Results: {sq['results_count']}")
        print()

In [ ]:
# Visualize multi-hop trace if available
if metadata.get('multi_hop_trace'):
    print("🔄 Multi-hop Reasoning Trace:\n")
    for step in metadata['multi_hop_trace']:
        print(f"  Iteration {step['iteration']}:")
        print(f"    Query: {step['query']}")
        print(f"    Reasoning: {step['reasoning']}")
        print(f"    Sources found: {step['sources_found']}")
        print()
else:
    print("ℹ️ No multi-hop trace available for this query")

# Show activity log summary
if metadata.get('activity_log'):
    print(f"📋 Activity Log ({len(metadata['activity_log'])} steps):")
    for activity in metadata['activity_log'][:5]:
        print(f"  - {activity}")

## 8. Exercise 7: Multimodal Queries (Figures & Tables)

Query for visual content and inspect how figures are retrieved with SAS URLs.

In [ ]:
# Query specifically for diagrams/figures
figure_question = "Show me the system layout diagrams and zoning maps"

result = execute_query(figure_question, config={
    "content_type_filter": "figure",
    "top_k": 3
})

print(f"📊 Found {len(result['sources'])} figure(s):\n")

for i, source in enumerate(result['sources'], 1):
    print(f"Figure {i}:")
    print(f"  Document: {source['source_document']}")
    print(f"  Page: {source['page_numbers']}")
    print(f"  Description: {source['content'][:200]}...")
    print(f"  Score: {source['relevance_score']:.3f}")
    
    if source.get('image_sas_url'):
        print(f"  🖼️ Image URL available (expires in 1 hour)")
    print()

In [ ]:
# Display an image if available (requires IPython)
from IPython.display import Image, display

figure_sources = [s for s in result['sources'] if s.get('image_sas_url')]

if figure_sources:
    print("🖼️ Displaying first retrieved figure:")
    display(Image(url=figure_sources[0]['image_sas_url'], width=600))
else:
    print("ℹ️ No figures with images found in the results")

## 9. Summary & Next Steps

### What You Learned

1. **Document Processing**: How documents are ingested, chunked, and indexed
2. **Index Schema**: The structure of Azure AI Search index for RAG
3. **Retrieval Strategies**: Comparing hybrid vs. agentic approaches
4. **Parameter Tuning**: Effect of top_k, filters, and search modes
5. **Query Decomposition**: How AI Agents break down complex queries
6. **Multimodal RAG**: Handling figures and tables with SAS URLs

### Next Steps

- Explore the React UI at http://localhost:5173
- Try Hebrew/RTL content to test internationalization
- Experiment with GraphRAG for cross-document reasoning
- Review the backend services code to understand the implementation

In [ ]:
# Cleanup - close the HTTP client
client.close()
print("✅ Lab complete! HTTP client closed.")